# Preparación y revisión de las bases históricas

Este notebook organiza los archivos locales de `BASES`; no formula ni resuelve el MIP. Su función es dejar disponibles las temporadas históricas de la Premier League y comprobar su estructura antes de usarlas en las instancias.

## Apartados

- Localiza `BASES/repository.xml` y lee el índice de instancias RobinX para mostrar qué casos cataloga.
- Carga los CSV de 2021-2022 a 2026-2027, comprueba las columnas `Date`, `HomeTeam` y `AwayTeam`, elimina filas incompletas y añade la etiqueta de temporada.
- Resume cantidad de partidos, equipos detectados y si cada base alcanza 380 partidos; consolida los partidos en `df_matches` y conserva cada temporada en `season_dataframes`.

La clasificación de 380 partidos es un control básico de volumen, no una auditoría de que el calendario histórico cumpla todas las reglas ni de que los datos estén actualizados. Para construir un calendario optimizado, los notebooks de `base_<temporada>.ipynb` pasan los equipos al modelo compartido.

In [3]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

# Localiza BASES desde la carpeta actual del notebook o cualquiera de sus padres.
current = Path.cwd().resolve()
base_dir = next(
    (
        parent / "BASES"
        for parent in (current, *current.parents)
        if (parent / "BASES" / "repository.xml").is_file()
    ),
    None,
)
if base_dir is None:
    raise FileNotFoundError(
        f"No se encontró BASES/repository.xml desde el directorio {current}"
    )

# Leer el índice RobinX. Este XML cataloga instancias de calendarios; los CSV
# de temporadas recientes están almacenados aparte en la misma carpeta BASES.
repository_path = base_dir / "repository.xml"
root = ET.parse(repository_path).getroot()
repository_instances = []
for category in root:
    for member in category.findall("member"):
        instance_node = member.find("instanceFile")
        if instance_node is not None and instance_node.text:
            instance_path = instance_node.text.strip()
            repository_instances.append(
                {
                    "category": category.tag,
                    "path": instance_path,
                    "name": Path(instance_path).stem,
                }
            )

print(f"Índice repository.xml leído: {repository_path}")
print(f"Instancias indexadas: {len(repository_instances)}")

# Cargar y clasificar separadamente cada base de temporada, desde 2021-2022
# hasta 2026-2027. Cada DataFrame conserva fecha, local y visitante.
seasons = [f"{year}-{year + 1}" for year in range(2021, 2027)]
required_columns = ["Date", "HomeTeam", "AwayTeam"]
season_dataframes = {}
summary_rows = []

for season in seasons:
    csv_path = base_dir / f"{season}.csv"
    if not csv_path.is_file():
        summary_rows.append(
            {
                "Temporada": season,
                "Archivo": csv_path.name,
                "Partidos cargados": 0,
                "Equipos": 0,
                "Clasificación": "Archivo no encontrado",
            }
        )
        continue

    df_season = pd.read_csv(csv_path)
    missing_columns = [
        column for column in required_columns if column not in df_season.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Faltan columnas en {csv_path.name}: {missing_columns}. "
            f"Columnas presentes: {list(df_season.columns)}"
        )

    df_season = df_season[required_columns].dropna().copy()
    df_season["HomeTeam"] = df_season["HomeTeam"].astype(str).str.strip()
    df_season["AwayTeam"] = df_season["AwayTeam"].astype(str).str.strip()
    df_season["Season"] = season
    season_dataframes[season] = df_season

    match_count = len(df_season)
    team_count = pd.unique(
        pd.concat([df_season["HomeTeam"], df_season["AwayTeam"]], ignore_index=True)
    ).size
    status = "Completa (380 partidos)" if match_count >= 380 else "Parcial o en curso"
    summary_rows.append(
        {
            "Temporada": season,
            "Archivo": csv_path.name,
            "Partidos cargados": match_count,
            "Equipos": team_count,
            "Clasificación": status,
        }
    )

# Tabla resumen y un DataFrame consolidado para análisis entre temporadas.
season_summary = pd.DataFrame(summary_rows)
df_matches = (
    pd.concat(season_dataframes.values(), ignore_index=True)
    if season_dataframes
    else pd.DataFrame(columns=required_columns + ["Season"])
)

print("\nResumen de bases de datos por temporada:")
display(season_summary)
print(f"\nTotal de partidos cargados: {len(df_matches)}")
print("Ejemplo de partidos:")
display(df_matches.head())

# Acceso a una temporada particular, por ejemplo:
# df_2023_2024 = season_dataframes["2023-2024"]

Índice repository.xml leído: C:\Users\camip\OneDrive\Escritorio\Premier League OPTI II\Premier-League\BASES\repository.xml
Instancias indexadas: 1385

Resumen de bases de datos por temporada:


,Temporada,Archivo,Partidos cargados,Equipos,Clasificación
0,2021-2022,2021-2022.csv,380,20,Completa (380 partidos)
1,2022-2023,2022-2023.csv,380,20,Completa (380 partidos)
2,2023-2024,2023-2024.csv,380,20,Completa (380 partidos)
3,2024-2025,2024-2025.csv,380,20,Completa (380 partidos)
4,2025-2026,2025-2026.csv,380,20,Completa (380 partidos)
5,2026-2027,2026-2027.csv,50,20,Parcial o en curso



Total de partidos cargados: 1950
Ejemplo de partidos:


,Date,HomeTeam,AwayTeam,Season
0,13/08/2021,Brentford,Arsenal,2021-2022
1,14/08/2021,Man United,Leeds,2021-2022
2,14/08/2021,Burnley,Brighton,2021-2022
3,14/08/2021,Chelsea,Crystal Palace,2021-2022
4,14/08/2021,Everton,Southampton,2021-2022
